In [ ]:
import illustris_python as il
import numpy as np
import h5py
import logging
import gc
from scipy import interpolate
from astropy.cosmology import Planck15 as cosmo

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("process_galaxies.log"), logging.StreamHandler()]
)

class Subhalo(object):
    def __init__(self, basePath, snapshot, subhaloID):
        self.basePath = basePath
        self.snapshot = snapshot
        self.id = subhaloID
        self.boxSize = 35000.0  # ckpc/h
        
        # 获取 Time (scale factor a), Redshift, Hubble para
        header = il.groupcat.loadHeader(basePath, snapshot)
        self.a = header['Time']
        self.z = header['Redshift']
        self.h = header['HubbleParam']
        
        # 加载 Subhalo 基础属性
        subhalo = il.groupcat.loadSingle(basePath, snapshot, subhaloID=self.id)
        self.subpos = subhalo['SubhaloPos']    # ckpc/h
        self.subvel = subhalo['SubhaloVel']    # km/s
        self.rHalfStars = subhalo['SubhaloHalfmassRadType'][4] # ckpc/h
        del subhalo
        gc.collect()

    def apply_pbc(self, coords):
        """ 处理周期性边界条件并平移至中心 (ckpc/h) """
        dx = coords - self.subpos
        halfBox = self.boxSize / 2.0
        return ((dx + halfBox) % self.boxSize) - halfBox

    def loadTNG(self, partType, fields):
        data = il.snapshot.loadSubhalo(self.basePath, self.snapshot, self.id, partType, fields=fields)
        # 如果只请求了一个字段，data 会是 ndarray，手动把它包装回 dict
        if len(fields) == 1 and not isinstance(data, dict):
            return {fields[0]: data}
        return data

class ProcessGasData(Subhalo):
    @staticmethod    
    def utherm_ne_to_temp(utherm, nelec):
        """ 
        Convert the InternalEnergy and ElectronAbundance of gas cells to temperature in [K].
        """
        hydrogen_massfrac = 0.76
        mass_proton = 1.672622e-24  # cgs
        gamma = 5/3
        boltzmann = 1.380650e-16  # cgs (erg/K)
        
        # calculate mean molecular weight
        meanmolwt = 4.0 / (1.0 + 3.0 * hydrogen_massfrac + 4.0 * hydrogen_massfrac * nelec) * mass_proton
        # return temperature (units in [K])
        return utherm * (gamma-1.0) / boltzmann * 1e10 * meanmolwt

    def is_ISM(self, temperature, density):
        log_temp = np.log10(temperature)
        condi = 6 + 0.25 * np.log10(density / (self.a**3))
        return np.where(log_temp <= condi, temperature, 1e8)

    def process_gas(self, h_group):
        # 1. 计算旋转矩阵 (仅读取计算 I 必须的字段)
        data = self.loadTNG('gas', ['Coordinates', 'Masses', 'StarFormationRate'])
        dx_ckpc = self.apply_pbc(data['Coordinates'])
        
        # 筛选 2*rHalf 内的有恒星形成的气体计算惯性张量
        dist = np.sqrt(np.sum(dx_ckpc**2, axis=1))
        mask_rot = (dist <= 2.0 * self.rHalfStars) & (data['StarFormationRate'] > 0.0)
        if np.sum(mask_rot) < 32: mask_rot = (dist <= 2.0 * self.rHalfStars)
        
        m, xyz = data['Masses'][mask_rot], dx_ckpc[mask_rot]
        I = np.zeros((3, 3), dtype='float32')
        I[0,0] = np.sum(m * (xyz[:,1]**2 + xyz[:,2]**2))
        I[1,1] = np.sum(m * (xyz[:,0]**2 + xyz[:,2]**2))
        I[2,2] = np.sum(m * (xyz[:,0]**2 + xyz[:,1]**2))
        I[0,1] = I[1,0] = -np.sum(m * (xyz[:,0] * xyz[:,1]))
        I[0,2] = I[2,0] = -np.sum(m * (xyz[:,0] * xyz[:,2]))
        I[1,2] = I[2,1] = -np.sum(m * (xyz[:,1] * xyz[:,2]))
        
        eigen_values, eigen_vectors = np.linalg.eigh(I)
        
        idx = np.argsort(eigen_values)
        i_min, i_mid, i_max = idx[0], idx[1], idx[2]
        
        x_axis = eigen_vectors[:, i_min]  # 长轴
        y_axis = eigen_vectors[:, i_mid]  # 短轴
        z_axis = eigen_vectors[:, i_max]  # 法向

        # 右手系
        if np.dot(np.cross(x_axis, y_axis), z_axis) < 0:
            x_axis = -x_axis

        # 构造旋转矩阵（行 = 新坐标轴）
        rot_matrix = np.vstack([x_axis, y_axis, z_axis])
        
        # 2. 确定最终筛选 Mask (基于旋转后的物理坐标)
        coords_phys = np.dot(dx_ckpc, rot_matrix.T) * (self.a / self.h)
        mask = (np.abs(coords_phys[:,0]) < 55) & (np.abs(coords_phys[:,1]) < 55) & (np.abs(coords_phys[:,2]) < 55)
        
        # 写入坐标
        h_group.create_dataset('x', data=coords_phys[mask, 0].astype('f4'))
        h_group.create_dataset('y', data=coords_phys[mask, 1].astype('f4'))
        h_group.create_dataset('z', data=coords_phys[mask, 2].astype('f4'))
        
        del data, dx_ckpc, coords_phys, dist, mask_rot
        gc.collect()

        # 3. 逐个/成对加载其余字段并写入
        # 密度与金属度
        data = self.loadTNG('gas', ['Density', 'GFM_Metallicity'])
        dens_mult = (1e10 * self.h**2) / (self.a**3 * 1e9)
        h_group.create_dataset('density', data=(data['Density'][mask] * dens_mult).astype('f4'))  # M_sun/pc3
        h_group.create_dataset('metallicity', data=data['GFM_Metallicity'][mask].astype('f4'))
        del data
        gc.collect()

        # 温度 (需要 Energy 和 Electron)
        data = self.loadTNG('gas', ['InternalEnergy', 'ElectronAbundance', 'Density'])
        temp = self.utherm_ne_to_temp(data['InternalEnergy'][mask], data['ElectronAbundance'][mask])
        temp = self.is_ISM(temp, data['Density'][mask])
        h_group.create_dataset('temperature', data=temp.astype('f4'))
        del data, temp
        gc.collect()

        # 速度
        vels = self.loadTNG('gas', ['Velocities'])['Velocities'][mask]
        vel_phys = vels * np.sqrt(self.a) - self.subvel
        vel_rot = np.dot(vel_phys, rot_matrix.T)
        h_group.create_dataset('vel_x', data=vel_rot[:,0].astype('f4'))
        h_group.create_dataset('vel_y', data=vel_rot[:,1].astype('f4'))
        h_group.create_dataset('vel_z', data=vel_rot[:,2].astype('f4'))
        del vels, vel_phys, vel_rot
        gc.collect()

        return rot_matrix

class ProcessStarData(Subhalo):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # 年龄插值准备
        self.age_snap = cosmo.age(self.z).value  # Universe Age(Gyr) at current snap
        a_samples = np.linspace(0.01, self.a, 50)
        age_samples = self.age_snap - cosmo.age(1/a_samples - 1).value
        
        a_samples[-1] = self.a
        age_samples[-1] = 0.0
        self.age_func = interpolate.interp1d(
            a_samples, age_samples, 
            kind='cubic', 
            fill_value="extrapolate"
        )

    def process_star(self, h_group, rot_matrix):
        # 1. 加载位置并筛选
        data = self.loadTNG('star', ['Coordinates', 'GFM_StellarFormationTime'])
        # 排除风粒子 (FormationTime > 0)
        is_star = data['GFM_StellarFormationTime'] > 0
        
        dx_ckpc = self.apply_pbc(data['Coordinates'][is_star])
        coords_phys = np.dot(dx_ckpc, rot_matrix.T) * (self.a / self.h)
        mask = (np.abs(coords_phys[:,0]) < 55) & (np.abs(coords_phys[:,1]) < 55) & (np.abs(coords_phys[:,2]) < 55)
        
        h_group.create_dataset('x', data=coords_phys[mask, 0].astype('f4'))  # kpc
        h_group.create_dataset('y', data=coords_phys[mask, 1].astype('f4'))
        h_group.create_dataset('z', data=coords_phys[mask, 2].astype('f4'))
        
        # 保存 mask 对应的原始索引用于后续读取其他字段
        # 注意：这里的 mask 是相对于 is_star 后的数组的
        
        # 处理年龄
        form_a = data['GFM_StellarFormationTime'][is_star][mask]
        # 即使某些粒子的 form_a 稍微超过了 self.a (比如 1.0000001 vs 0.9999999)
        # 我们也强制将其拉回到 self.a，防止插值产生负数年龄
        form_a_safe = np.minimum(form_a, self.a)
        
        star_ages = self.age_func(form_a_safe)
        
        # 再次确认物理下限：最小年龄设为 100 年 (1e-7 Gyr)
        star_ages = np.maximum(star_ages, 1e-7)
        h_group.create_dataset('age', data=star_ages.astype('f4'))
        
        del data, dx_ckpc, coords_phys, form_a, form_a_safe, star_ages
        gc.collect()

        # 2. 质量、金属度、Hsml
        data = self.loadTNG('star', ['GFM_InitialMass', 'GFM_Metallicity', 'StellarHsml'])
        h_group.create_dataset('init_mass', data=(data['GFM_InitialMass'][is_star][mask] * 1e10 / self.h).astype('f4'))  # Msun
        h_group.create_dataset('metallicity', data=data['GFM_Metallicity'][is_star][mask].astype('f4'))  # 1
        hsml_phys = (data['StellarHsml'][is_star][mask] * self.a / self.h).clip(max=0.8)  # kpc
        h_group.create_dataset('hsml', data=hsml_phys.astype('f4'))
        del data
        gc.collect()

        # 3. 速度
        vels = self.loadTNG('star', ['Velocities'])['Velocities'][is_star][mask]
        vel_phys = vels * np.sqrt(self.a) - self.subvel  # km/s
        vel_rot = np.dot(vel_phys, rot_matrix.T)
        h_group.create_dataset('vel_x', data=vel_rot[:,0].astype('f4'))
        h_group.create_dataset('vel_y', data=vel_rot[:,1].astype('f4'))
        h_group.create_dataset('vel_z', data=vel_rot[:,2].astype('f4'))
        del vels, vel_phys, vel_rot
        gc.collect()

        
def main(outputfile, ids):
    basePath = '/home/tnguser/sims.TNG/TNG50-1/output/'
    snapshot = 33  # 不同红移要调整这里
    
    with h5py.File(outputfile, 'w') as hdf:
        for i, sid in enumerate(ids):
            logging.info(f"Processing ID {sid} ({i+1}/{len(ids)})")
            
            try:
                gal_grp = hdf.create_group(f"Sub{sid}")
                gas_grp = gal_grp.create_group('gas')
                star_grp = gal_grp.create_group('star')
                
                # 1. 处理 Gas 并获取旋转矩阵
                gas_obj = ProcessGasData(basePath, snapshot, sid)
                rot_matrix = gas_obj.process_gas(gas_grp)
                
                # 2. 处理 Star (共享旋转矩阵)
                star_obj = ProcessStarData(basePath, snapshot, sid)
                star_obj.process_star(star_grp, rot_matrix)
                
                hdf.flush() 
                
                # 显式清理
                del gas_obj, star_obj
                gc.collect()
                
            except Exception as e:
                logging.error(f"Error processing ID {sid}: {e}")
                if str(sid) in hdf: del hdf[str(sid)]

if __name__ == "__main__":
    outputfile = "/path/to/where/you/want/to/save/yourdata.hdf5"
    ID_file = '/path/to/GalaxyIDs.txt'
    ids = np.loadtxt(ID_file, dtype=int)
    ids = np.array([107194,])
    # 如果 ids 是标量，转为 list
    if ids.ndim == 0: ids = [int(ids)]
    main(outputfile, ids)

2026-04-10 10:53:00,950 - INFO - Processing ID 107194 (1/1)
